# Non-Contractual Player Churn Analytics & Predictive Pipeline

## 1. Business Understanding & Problem Formulation

### 1.1 Context & Industry Economics
In online gaming and wagering platforms, customer acquisition costs (CAC) typically exceed retention costs by 4x to 7x. Unlike subscription-based platforms (e.g., Netflix, Spotify), player churn is **non-contractual**:
- Customers never file formal account cancellations.
- Churn manifests as silent disengagement, balance drawdown, or migration to competitor sportsbooks and poker rooms.

### 1.2 Business Objective & Decision Support
Build an end-to-end data pipeline and classification model to:
1. Identify behavioral warning signals (disengagement velocity, tilt, cashier friction) prior to permanent account abandonment.
2. Flag players at risk of entering a **45-day dormancy window** following an observation cutoff date ($T_{\text{cutoff}}$).
3. Deliver an interpretable score (risk probability) enabling automated CRM re-engagement campaigns before players churn permanently.

### 1.3 Analytical Methodology (CRISP-DM Framework)
1. **Business Understanding**: Define non-contractual retention mechanics and economics.
2. **Data Understanding & Hygiene**: Audit schema relationships, timestamp alignments, and key integrity.
3. **Exploratory Data Analysis (EDA)**:
   - *Global Univariate Analysis*: Distributional skewness, heavy tails, and categorical cardinality.
   - *Bivariate & Temporal Dynamics*: Macro volume growth, seasonality, and product-to-cashier interactions.
4. **Temporal Framing & Target Engineering**: Establish $T_{\text{cutoff}}$ to label ground-truth churn without lookahead bias.
5. **Behavioral Feature Engineering**: Construct player-level RFM, velocity ratios, and tilt metrics.
6. **Cohort Bivariate & Multivariate Diagnostics**: Statistical contrasts across churn cohorts and multicollinearity auditing.
7. **Predictive Modeling & Cost-Benefit Evaluation**: Benchmark classification algorithms and tune decision thresholds against business retention economics.

## 2. Data Understanding & Structural Validation

### 2.1 Entity Schema & Source Profiles
The analytical dataset integrates five primary relational tables from the bwin longitudinal dataset:
- `Demographics`: Baseline account attributes (`SystemAgeAsOfReg`, `Gender`, `CountryID`).
- `CashGames`: Session frequencies and stakes for ring games (`Windows`, `StakesC`, `WinningsC`).
- `Tournaments`: Discrete tournament participation (`Trnmnts`, `StakesT`, `WinningsT`).
- `Deposits`: Liquidity injection events, gateway routing, and processing outcomes (`Amount`, `Status`).
- `Withdrawals`: Capital extraction events and payment outcomes (`Amount`, `Status`).

### 2.2 Integrity Objectives
- Verify data types, casting date strings into unified `datetime64[ns]` formats.
- Confirm primary key integrity (`UserID`) to determine whether active gaming accounts map cleanly to demographic baselines without orphan records.

In [1]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

In [2]:
# Loading datasets
Demographics =pd.read_csv('datasets/SecondSeshData1Demographics.csv') 
CashGames = pd.read_csv('datasets/SecondSeshData2CashGames.csv')
Tournaments = pd.read_csv('datasets/SecondSeshData3Tournaments.csv')
Deposits = pd.read_csv('datasets/SecondSeshData4Deposits.csv')
Withdrawals = pd.read_csv('datasets/SecondSeshData5Withdrawals.csv')

In [13]:
# Find the data types of each date column in the tables
table_names = ("CashGames", "Tournaments", "Deposits", "Withdrawals")

for name, table in zip(
    table_names,
    (CashGames, Tournaments, Deposits, Withdrawals)
):
    print(f"{name}:")
    print(table.dtypes)
    print()


CashGames:
UserID         int64
Date             str
Windows        int64
StakesC      float64
WinningsC    float64
dtype: object

Tournaments:
UserID         int64
Date             str
Trnmnts        int64
StakesT      float64
WinningsT    float64
dtype: object

Deposits:
UserID           int64
DepositID        int64
SummaryDate        str
ProcessDate        str
ProcessTime        str
PayMeth            str
PayMethCat         str
CardType           str
Amount         float64
Status             str
dtype: object

Withdrawals:
UserID            int64
WithdrawalID      int64
SummaryDate         str
ProcessDate         str
ProcessTime         str
PayMeth             str
PayMethCat          str
CardType            str
Amount          float64
Status              str
dtype: object



In [21]:
# Are there any missing values in the tables?
tables = (Demographics, CashGames, Tournaments, Deposits, Withdrawals)
for name, table in zip(
    ("Demographics", "CashGames", "Tournaments", "Deposits", "Withdrawals"),
    tables
):
    #Sum up nulls in all columns of the table
    null_counts = table.isna().sum()
    print(f"{name}:")
    print(null_counts)

Demographics:
UserID               0
SystemAgeAsOfReg     0
Gender               0
CountryID           12
dtype: int64
CashGames:
UserID       0
Date         0
Windows      0
StakesC      0
WinningsC    0
dtype: int64
Tournaments:
UserID       0
Date         0
Trnmnts      0
StakesT      0
WinningsT    0
dtype: int64
Deposits:
UserID            0
DepositID         0
SummaryDate       0
ProcessDate       0
ProcessTime       0
PayMeth           0
PayMethCat        0
CardType       9920
Amount            0
Status            0
dtype: int64
Withdrawals:
UserID             0
WithdrawalID       0
SummaryDate        0
ProcessDate        0
ProcessTime        0
PayMeth            0
PayMethCat         0
CardType        1685
Amount             0
Status             0
dtype: int64


In [15]:
Demographics.isna().sum()

UserID               0
SystemAgeAsOfReg     0
Gender               0
CountryID           12
dtype: int64

In [16]:
Withdrawals.isna().sum()

UserID             0
WithdrawalID       0
SummaryDate        0
ProcessDate        0
ProcessTime        0
PayMeth            0
PayMethCat         0
CardType        1685
Amount             0
Status             0
dtype: int64

In [18]:
Withdrawals['CardType'].value_counts()

CardType
Und    30622
Name: count, dtype: int64

In [19]:
Withdrawals.shape

(32307, 10)